# Genotype–Phenotype Heterogeneity: FAIR^2 Dataset Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the FAIR^2 clinical dataset for non-classical 21-hydroxylase deficiency using the `mlcroissant` library.

### Dataset Source
The dataset Croissant schema is provided at:
https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json

In [ ]:
# Install mlcroissant if needed
!pip install mlcroissant

## 1. Data Loading

We start by loading the dataset metadata from the Croissant schema URL using `mlcroissant`. This allows us to inspect general information about the dataset such as its name and description.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json"

# Load the dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)

# Access metadata as an object
metadata = dataset.metadata

print(f"Name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Identifier: {metadata.identifier}")

## 2. Data Overview

Examine available record sets and fields. Each record set and field in Croissant is identified by a unique `@id`.

We list the record sets and their fields by referencing their respective `@id` values.

In [ ]:
# Show available record sets and their field @id's
record_sets = dataset.record_sets
print("Available record sets:")
for rs in record_sets:
    print(f"RecordSet @id: {rs['@id']}")
    print(f"  Name: {rs.get('name', 'N/A')}")
    print("  Fields:")
    for field in rs.get('field', []):
        print(f"    Field @id: {field['@id']}, Name: {field.get('name', 'N/A')}")
    print()

# List sample records for each record set, referencing the @id
for rs in record_sets:
    print(f"Sample records for RecordSet @id: {rs['@id']}")
    try:
        for i, record in enumerate(dataset.records(record_set=rs['@id'])):
            print(record)
            if i >= 2:
                break
    except Exception as e:
        print("No records found or error:", e)
    print("---")

## 3. Data Extraction

Load data from each record set into pandas DataFrames.

We dynamically reference record set `@id`s and extract data for further analysis.

In [ ]:
# Extract data from all record sets using their @id
dataframes = {}

for rs in record_sets:
    rs_id = rs['@id']
    try:
        records = list(dataset.records(record_set=rs_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"Loaded DataFrame for RecordSet @id: {rs_id}")
            print(f"Columns: {df.columns.tolist()}")
            print(df.head(), "\n")
        else:
            print(f"No records for RecordSet @id: {rs_id}")
    except Exception as e:
        print(f"Failed to load records for @id {rs_id}: {e}")

## 4. Exploratory Data Analysis (EDA)

Perform common data processing steps (filtering, normalization, grouping) referencing fields by their `@id`s.

Below, we select a numeric field for analysis, filter for values above a threshold, normalize, and group by a categorical field.

In [ ]:
# EDA: Choose a RecordSet with numeric data for a demonstration
# We'll choose the first record set with numeric field
selected_rs_id = None
numeric_field_id = None
group_field_id = None

for rs in record_sets:
    rs_id = rs['@id']
    df = dataframes.get(rs_id)
    if df is not None:
        numeric_candidates = [field['@id'] for field in rs.get('field', []) 
                             if field.get('dataType') in ['schema:Float', 'schema:Integer', 'schema:Number']]
        group_candidates = [field['@id'] for field in rs.get('field', [])
                           if field.get('dataType') == 'schema:Text']
        # Use the first numeric field and first group field with records
        if numeric_candidates:
            selected_rs_id = rs_id
            numeric_field_id = numeric_candidates[0]
            if group_candidates:
                group_field_id = group_candidates[0]
            break

if selected_rs_id and numeric_field_id:
    df = dataframes[selected_rs_id].copy()
    # Filter: keep values above threshold for numeric_field
    threshold = 10
    if numeric_field_id in df.columns:
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
            filtered_df[numeric_field_id].std()
        )
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by the group_field_id
        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped data by {group_field_id} (mean of {numeric_field_id}):")
            print(grouped_df.head())
    else:
        print(f"Numeric field {numeric_field_id} not found in DataFrame columns.")
else:
    print("No suitable record set and numeric field found for EDA demonstration.")

## 5. Visualization

Visualize the distribution of the selected numeric field. Also, visualize relationships to the group field if available.

In [ ]:
# Visualization
if selected_rs_id and numeric_field_id:
    df = dataframes[selected_rs_id]
    if numeric_field_id in df.columns:
        plt.figure(figsize=(8,5))
        df[numeric_field_id].hist(bins=10)
        plt.title(f"Distribution of {numeric_field_id} in RecordSet {selected_rs_id}")
        plt.xlabel(numeric_field_id)
        plt.ylabel("Count")
        plt.show()

        if group_field_id and group_field_id in df.columns:
            plt.figure(figsize=(8,5))
            df.groupby(group_field_id)[numeric_field_id].mean().plot(kind="bar")
            plt.title(f"Mean {numeric_field_id} by {group_field_id} in RecordSet {selected_rs_id}")
            plt.xlabel(group_field_id)
            plt.ylabel(f"Mean {numeric_field_id}")
            plt.show()
    else:
        print(f"Numeric field {numeric_field_id} not found for visualization.")
else:
    print("No suitable numeric data found for visualization.")

## 6. Conclusion

In this notebook, we:
- Used `mlcroissant` to load metadata and structured records from the FAIR^2 dataset.
- Explored the schema, listing all record sets and fields referenced by their `@id`.
- Loaded tabular records for analysis and performed basic EDA (filtering, normalization, grouping) referencing Croissant entity IDs.
- Visualized data distributions, grouping by categorical descriptors when available.

The FAIR^2 dataset provides rich, clearly labeled clinical, laboratory, imaging, and genetic information for three patients diagnosed with non-classical 21-hydroxylase deficiency-associated infertility. The small sample size and specific context underscore the need for careful interpretation and caution in generalization.